# 🔬 Autonomous Research Assistant - ML Training Pipeline

**Project:** Autonomous Research Assistant 
**Team:** Gnaneshwar Reddy Dontireddy, Yedukondalu Reddy Degala 
**Department:** CSE, Vignan's Foundation for Science, Technology & Research 

**Dataset:** Cornell University arXiv Papers (Kaggle) 
**Objective:** Train a text classifier to categorize research papers into domains based on abstracts

---

## 1. Install Dependencies & Setup

In [ ]:
!pip install -q kagglehub wordcloud nltk scikit-learn seaborn matplotlib pandas numpy

In [ ]:
import json
import os
import re
import pickle
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score
)

warnings.filterwarnings('ignore')

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print('✅ All libraries loaded successfully!')

---
## 2. Download & Load arXiv Dataset

In [ ]:
import kagglehub

# Download the arXiv dataset from Kaggle
path = kagglehub.dataset_download("Cornell-University/arxiv")
DATA_FILE = os.path.join(path, "arxiv-metadata-oai-snapshot.json")
print(f"Dataset path: {DATA_FILE}")
print(f"File exists: {os.path.exists(DATA_FILE)}")

In [ ]:
# Define target research domains
TARGET_CATEGORIES = {
    'cs.AI': 'Artificial Intelligence',
    'cs.CL': 'Natural Language Processing',
    'cs.CV': 'Computer Vision',
    'cs.LG': 'Machine Learning',
    'stat.ML': 'Statistical ML',
    'physics.comp-ph': 'Computational Physics'
}

SAMPLES_PER_CATEGORY = 2000

print(f"Categories: {list(TARGET_CATEGORIES.values())}")
print(f"Samples per category: {SAMPLES_PER_CATEGORY}")
print(f"Total expected samples: {SAMPLES_PER_CATEGORY * len(TARGET_CATEGORIES)}")

---
## 3. Data Extraction

In [ ]:
%%time

papers = []
category_counts = Counter()

with open(DATA_FILE, 'r') as f:
    for line_num, line in enumerate(f):
        # Stop when we have enough samples for all categories
        if all(category_counts[c] >= SAMPLES_PER_CATEGORY for c in TARGET_CATEGORIES):
            break

        try:
            paper = json.loads(line)
            categories = paper.get('categories', '').split()

            for cat in categories:
                if cat in TARGET_CATEGORIES and category_counts[cat] < SAMPLES_PER_CATEGORY:
                    papers.append({
                        'title': paper.get('title', '').replace('\n', ' ').strip(),
                        'abstract': paper.get('abstract', '').replace('\n', ' ').strip(),
                        'category': cat,
                        'category_name': TARGET_CATEGORIES[cat]
                    })
                    category_counts[cat] += 1
                    break
        except json.JSONDecodeError:
            continue

        if line_num % 100000 == 0:
            print(f"  Scanned {line_num:,} lines... Collected: {dict(category_counts)}")

df = pd.DataFrame(papers)
print(f"\n✅ Extracted {len(df)} papers")
print(f"\nDistribution:")
print(df['category_name'].value_counts())

In [ ]:
# Preview the data
df.head(10)

---
## 4. Text Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """Clean and preprocess text for ML training."""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

print("Preprocessing abstracts...")
df['cleaned_abstract'] = df['abstract'].apply(preprocess_text)
df['cleaned_title'] = df['title'].apply(preprocess_text)
df['text'] = df['cleaned_title'] + ' ' + df['cleaned_abstract']

# Encode labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['category'])

print(f"\n✅ Preprocessing complete!")
print(f"Avg word count (cleaned): {df['cleaned_abstract'].str.split().str.len().mean():.0f}")
print(f"Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

In [ ]:
# Show before vs after preprocessing
sample = df.iloc[0]
print("ORIGINAL ABSTRACT:")
print(sample['abstract'][:300])
print("\nCLEANED ABSTRACT:")
print(sample['cleaned_abstract'][:300])

---
## 5. Data Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Category distribution
cat_counts = df['category_name'].value_counts()
colors = ['#6366F1', '#8B5CF6', '#10B981', '#F59E0B', '#EF4444', '#3B82F6']
axes[0].pie(cat_counts.values, labels=cat_counts.index, autopct='%1.1f%%', colors=colors)
axes[0].set_title('Dataset Distribution by Category', fontsize=13, fontweight='bold')

# Word count distribution
df['word_count'] = df['cleaned_abstract'].str.split().str.len()
axes[1].hist(df['word_count'], bins=50, color='#6366F1', alpha=0.7, edgecolor='white')
axes[1].set_xlabel('Word Count (after preprocessing)')
axes[1].set_ylabel('Number of Papers')
axes[1].set_title('Abstract Length Distribution', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('data_distribution.png', dpi=150)
plt.show()
print('📊 Saved: data_distribution.png')

In [ ]:
from wordcloud import WordCloud

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (cat, cat_name) in enumerate(TARGET_CATEGORIES.items()):
    text = ' '.join(df[df['category'] == cat]['cleaned_abstract'].values)
    wc = WordCloud(width=400, height=300, background_color='white',
                   colormap='viridis', max_words=50).generate(text)
    axes[idx].imshow(wc, interpolation='bilinear')
    axes[idx].set_title(cat_name, fontsize=12, fontweight='bold')
    axes[idx].axis('off')

plt.suptitle('Word Clouds per Research Domain', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('wordclouds.png', dpi=150)
plt.show()
print('📊 Saved: wordclouds.png')

---
## 6. TF-IDF Feature Extraction

In [ ]:
X = df['text'].fillna('')
y = df['label']

# 80-20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")

# TF-IDF Vectorization
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"\nTF-IDF matrix shape: {X_train_tfidf.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_)}")

---
## 7. Model Training & Comparison

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'Multinomial Naive Bayes': MultinomialNB(alpha=0.1),
    'Linear SVM': LinearSVC(max_iter=2000, C=1.0, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=50, random_state=42, n_jobs=-1)
}

results = {}
best_model_name = None
best_accuracy = 0

for name, model in models.items():
    print(f"\n🔄 Training: {name}...")
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')

    results[name] = {
        'accuracy': acc, 'f1_score': f1,
        'precision': prec, 'recall': rec,
        'predictions': y_pred, 'model': model
    }

    print(f"   Accuracy: {acc:.4f} | F1: {f1:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")

    if acc > best_accuracy:
        best_accuracy = acc
        best_model_name = name

print(f"\n🏆 Best Model: {best_model_name} (Accuracy: {best_accuracy:.4f})")

In [ ]:
# Model Comparison Chart
fig, ax = plt.subplots(figsize=(10, 6))
model_names = list(results.keys())
accuracies = [results[m]['accuracy'] for m in model_names]
f1_scores = [results[m]['f1_score'] for m in model_names]

x = np.arange(len(model_names))
width = 0.35

bars1 = ax.bar(x - width/2, accuracies, width, label='Accuracy', color='#6366F1')
bars2 = ax.bar(x + width/2, f1_scores, width, label='F1 Score', color='#10B981')

ax.set_ylabel('Score')
ax.set_title('Model Comparison - Research Paper Classification', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=15, ha='right')
ax.legend()
ax.set_ylim(0, 1.1)

for bar in bars1:
    ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)
for bar in bars2:
    ax.annotate(f'{bar.get_height():.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()
print('📊 Saved: model_comparison.png')

---
## 8. Best Model - Detailed Evaluation

In [ ]:
y_pred_best = results[best_model_name]['predictions']
category_names = list(le.classes_)

print(f"\n📋 Classification Report - {best_model_name}")
print("=" * 65)
print(classification_report(y_test, y_pred_best, target_names=category_names))

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 7))
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=category_names, yticklabels=category_names, ax=ax)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title(f'Confusion Matrix - {best_model_name}', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print('📊 Saved: confusion_matrix.png')

In [ ]:
# Per-class accuracy
fig, ax = plt.subplots(figsize=(10, 6))
per_class_acc = cm.diagonal() / cm.sum(axis=1)
colors = ['#6366F1', '#8B5CF6', '#10B981', '#F59E0B', '#EF4444', '#3B82F6']
bars = ax.barh(category_names, per_class_acc, color=colors[:len(category_names)])
ax.set_xlabel('Accuracy')
ax.set_title('Per-Category Classification Accuracy', fontsize=14, fontweight='bold')
ax.set_xlim(0, 1.1)

for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{acc:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('per_class_accuracy.png', dpi=150)
plt.show()
print('📊 Saved: per_class_accuracy.png')

In [ ]:
# Classification metrics heatmap
report = classification_report(y_test, y_pred_best, target_names=category_names, output_dict=True)
report_df = pd.DataFrame(report).transpose().iloc[:-3, :-1]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(report_df.astype(float), annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=['Precision', 'Recall', 'F1-Score'], ax=ax)
ax.set_title('Classification Metrics per Category', fontsize=14, fontweight='bold')
ax.set_ylabel('Category')
plt.tight_layout()
plt.savefig('classification_heatmap.png', dpi=150)
plt.show()
print('📊 Saved: classification_heatmap.png')

---
## 9. Custom Abstract Classification Test

In [ ]:
test_abstracts = [
    {
        "title": "Deep Reinforcement Learning for Robot Navigation",
        "abstract": "We present a deep reinforcement learning framework for autonomous robot navigation in unknown environments. Our approach uses a convolutional neural network to process raw sensor data and a policy gradient method to learn optimal navigation strategies.",
        "expected": "cs.AI"
    },
    {
        "title": "BERT-based Sentiment Analysis for Social Media",
        "abstract": "This paper proposes a fine-tuned BERT model for sentiment analysis on social media text. We address challenges including informal language, abbreviations, and emoji usage. Our approach achieves state-of-the-art performance on the SemEval benchmark.",
        "expected": "cs.CL"
    },
    {
        "title": "Real-time Object Detection using YOLOv5",
        "abstract": "We introduce an optimized YOLOv5 architecture for real-time object detection in autonomous vehicles. Our modifications to the backbone network reduce inference time by 40% while maintaining detection accuracy on KITTI and nuScenes datasets.",
        "expected": "cs.CV"
    },
    {
        "title": "Federated Learning with Differential Privacy",
        "abstract": "This work proposes a federated learning framework that incorporates differential privacy guarantees. We develop a novel gradient perturbation mechanism that provides epsilon-delta privacy while maintaining model convergence.",
        "expected": "cs.LG"
    },
    {
        "title": "Bayesian Optimization for Hyperparameter Tuning",
        "abstract": "We present a Gaussian process-based Bayesian optimization method for automatic hyperparameter tuning. Our approach uses an expected improvement acquisition function with a Matern kernel on 50 benchmark datasets.",
        "expected": "stat.ML"
    },
    {
        "title": "Quantum Monte Carlo Simulation of Hydrogen",
        "abstract": "We perform diffusion Monte Carlo simulations of liquid hydrogen at high pressure using a newly developed pseudopotential. Our calculations predict the metallization pressure with chemical accuracy compared to density functional theory.",
        "expected": "physics.comp-ph"
    }
]

best_model = results[best_model_name]['model']
correct = 0

print(f"Testing {len(test_abstracts)} custom abstracts with {best_model_name}:\n")

for i, test in enumerate(test_abstracts):
    cleaned = preprocess_text(test['abstract'])
    vec = tfidf.transform([cleaned])
    pred = best_model.predict(vec)[0]
    predicted_cat = le.classes_[pred]
    is_correct = predicted_cat == test['expected']
    correct += int(is_correct)

    status = '✅' if is_correct else '❌'
    print(f"{status} {test['title']}")
    print(f"   Expected: {test['expected']} → Predicted: {predicted_cat}\n")

print(f"\n📊 Custom Test Accuracy: {correct}/{len(test_abstracts)} ({correct/len(test_abstracts)*100:.0f}%)")

---
## 10. Feature Importance Analysis

In [ ]:
feature_names = tfidf.get_feature_names_out()
best_model = results[best_model_name]['model']

if hasattr(best_model, 'coef_'):
    print(f"Top 10 Keywords per Category ({best_model_name}):\n")
    for i, cat in enumerate(le.classes_):
        if i < best_model.coef_.shape[0]:
            top_idx = best_model.coef_[i].argsort()[-10:][::-1]
            top_words = [feature_names[j] for j in top_idx]
            print(f"  {cat} ({TARGET_CATEGORIES.get(cat, cat)}):")
            print(f"    {', '.join(top_words)}\n")
elif hasattr(best_model, 'feature_importances_'):
    top_idx = best_model.feature_importances_.argsort()[-20:][::-1]
    top_words = [feature_names[j] for j in top_idx]
    print(f"Top 20 features: {', '.join(top_words)}")

---
## 11. Save Model & Results

In [ ]:
# Save model and vectorizer
with open('best_model.pkl', 'wb') as f:
    pickle.dump(results[best_model_name]['model'], f)

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

# Save results summary
results_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results],
    'F1 Score': [results[m]['f1_score'] for m in results],
    'Precision': [results[m]['precision'] for m in results],
    'Recall': [results[m]['recall'] for m in results]
}).sort_values('Accuracy', ascending=False)

results_df.to_csv('model_results.csv', index=False)

print('✅ Saved: best_model.pkl')
print('✅ Saved: tfidf_vectorizer.pkl')
print('✅ Saved: model_results.csv')
print(f'\n{results_df.to_string(index=False)}')

---
## 12. Summary

| Artifact | Description |
|----------|-------------|
| `data_distribution.png` | Dataset category distribution & word count |
| `wordclouds.png` | Word clouds per research domain |
| `model_comparison.png` | Accuracy & F1 across 4 models |
| `confusion_matrix.png` | Best model confusion matrix |
| `per_class_accuracy.png` | Per-category accuracy bars |
| `classification_heatmap.png` | Precision/Recall/F1 heatmap |
| `model_results.csv` | All metrics in tabular form |
| `best_model.pkl` | Trained model (pickle) |
| `tfidf_vectorizer.pkl` | Fitted TF-IDF vectorizer |

**Dataset:** Cornell University arXiv (12,000 papers, 6 categories) 
**Best Model:** Determined at runtime based on accuracy 
**Pipeline:** Text Preprocessing → TF-IDF → Multi-model Training → Evaluation